In [37]:
spark

In [36]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

spark.sql("""
    CREATE TABLE IF NOT EXISTS bronze.orders_raw (
        source_system STRING,
        ingestion_time TIMESTAMP,
        raw_payload STRING,
        kafka_topic STRING,
        kafka_partition INT,
        kafka_offset BIGINT,
        kafka_timestamp TIMESTAMP,
        bronze_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/bronze/orders_raw'
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")


DataFrame[]

In [38]:
landing_stream = (
    spark.readStream
    .format("delta")
    .load("/opt/spark-data/delta/landing/raw_events")
)


In [39]:
from pyspark.sql.functions import col, current_timestamp, get_json_object

bronze_df = landing_stream.select(
    get_json_object(col("value"), "$.source_system").alias("source_system"),
    get_json_object(col("value"), "$.ingestion_time").cast("timestamp").alias("ingestion_time"),
    col("value").alias("raw_payload"),
    col("topic").alias("kafka_topic"),
    col("partition").alias("kafka_partition"),
    col("offset").alias("kafka_offset"),
    col("kafka_timestamp"),
    current_timestamp().alias("bronze_ingest_ts")
)


In [ ]:
bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .queryName("bronze_orders_raw_stream")
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark-data/checkpoints/bronze/v1/orders_raw")
    .trigger(processingTime="10 seconds")
    .start("/opt/spark-data/delta/bronze/orders_raw")
)

print("Bronze stream started...")


In [43]:
spark.sql("SELECT * FROM bronze.orders_raw").show(truncate=False)


+-------------+--------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [44]:
df = spark.table("bronze.orders_raw")


In [ ]:
df.select("raw_payload").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [45]:
for query in spark.streams.active:
    print(f"Query Name: {query.name}")
    print(f"Query ID:   {query.id}")
    print(f"Status:     {query.status['message']}")
    print("-" * 30)


Query Name: bronze_orders_raw_stream
Query ID:   d22a41c3-8b91-4555-9ef1-53b91566c15e
Status:     Waiting for next trigger
------------------------------
Query Name: landing_raw_events_stream
Query ID:   8687fd65-1931-485a-b202-4a1b9ae3b517
Status:     Waiting for next trigger
------------------------------


In [339]:
for query in spark.streams.active:
    query.stop()
    print(f"Stopped: {query.name}")
